# GPU Credit Scoring - Entraînement XGBoost GPU

**Dataset:** `dataset_scoring_benin_700k.csv` (700K lignes, 42 colonnes)
**Cible:** `defaut_paiement` (0 = rembourse, 1 = défaut)
**Objectif:** Entraîner XGBoost sur GPU (Tesla T4), comparer CPU/GPU, exporter le modèle

**Important:** Les features sont normalisées avant l'export. Le backend FastAPI applique la même normalisation.

In [ ]:
!pip install -q gdown

import gdown
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import time
import json

print('XGBoost version:', xgb.__version__)

In [ ]:
# Télécharger le dataset depuis Google Drive
FILE_ID = '17fIIactArdqnw9f7NSWgbQPyQYaioQo3'
OUTPUT = 'dataset_scoring_benin_700k.csv'

gdown.download(id=FILE_ID, output=OUTPUT, quiet=False)

df = pd.read_csv(OUTPUT)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Identifier la cible et les features
TARGET = 'defaut_paiement'
ID_COL = 'id_commercant'

CAT_COLS = ['genre', 'niveau_education', 'situation_familiale', 'departement', 'secteur_activite']
NUM_COLS = [c for c in df.columns if c not in [ID_COL, TARGET] + CAT_COLS]

print(f'Target: {TARGET}')
print(f'Target distribution:\n{df[TARGET].value_counts(normalize=True)}')
print(f'Numeric features: {len(NUM_COLS)}')
print(f'Categorical features: {len(CAT_COLS)}')

In [ ]:
# Encodage des variables catégorielles
df_encoded = pd.get_dummies(df, columns=CAT_COLS, drop_first=True)

# Features finales
exclude = [ID_COL, TARGET]
features = [c for c in df_encoded.columns if c not in exclude]

# Nettoyage
df_encoded = df_encoded.dropna(subset=[TARGET])
for col in features:
    if df_encoded[col].dtype == 'object':
        df_encoded[col] = df_encoded[col].fillna('unknown')
    else:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

X = df_encoded[features]
y = df_encoded[TARGET]
print(f'Features after encoding: {len(features)}')
print(f'Dataset shape: {X.shape}')

In [ ]:
# Normalisation - Log transform sur les montants
LOG_COLS = ['revenu_mensuel_declare', 'depenses_mensuelles', 'solde_moyen_compte',
            'volume_depots_moyen', 'volume_retraits_moyen', 'montant_demande',
            'montant_moyen_recharge', 'montant_moyen_cotisation']

for col in LOG_COLS:
    if col in X.columns:
        X[col] = np.log1p(X[col].clip(lower=0))

# Clip ratios entre 0 et 1
RATIO_COLS = ['tendance_solde_3m', 'regularite_depots_ecart_type',
              'part_transferts_marchands', 'stabilite_lieu_vie',
              'regularite_cotisations_tontine', 'score_aptitude_entrepreneuriale',
              'score_integrite', 'locus_controle_interne', 'garantie']

for col in RATIO_COLS:
    if col in X.columns:
        X[col] = X[col].clip(0, 1)

print('Features normalized')
print(X.describe())

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Entraînement CPU
model_cpu = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cpu',
    eval_metric='auc', random_state=42
)
t0 = time.time()
model_cpu.fit(X_train, y_train)
cpu_time = time.time() - t0
cpu_auc = roc_auc_score(y_test, model_cpu.predict_proba(X_test)[:, 1])
print(f'CPU: {cpu_time:.1f}s, AUC: {cpu_auc:.4f}')

In [ ]:
# Entraînement GPU
model_gpu = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda',
    eval_metric='auc', random_state=42
)
t0 = time.time()
model_gpu.fit(X_train, y_train)
gpu_time = time.time() - t0
gpu_auc = roc_auc_score(y_test, model_gpu.predict_proba(X_test)[:, 1])
speedup = cpu_time / gpu_time
print(f'GPU: {gpu_time:.1f}s, AUC: {gpu_auc:.4f}, Speedup: {speedup:.1f}x')

In [ ]:
# Export du modèle et métadonnées
model_gpu.save_model('xgboost_model.json')

with open('feature_order.json', 'w') as f:
    json.dump(features, f)

metadata = {
    'version': '1.0.0',
    'n_features': len(features),
    'dataset_size': len(df),
    'target': TARGET,
    'target_distribution': {str(k): float(v) for k, v in df[TARGET].value_counts(normalize=True).items()},
    'xgboost_version': xgb.__version__,
    'train_auc': float(roc_auc_score(y_train, model_gpu.predict_proba(X_train)[:, 1])),
    'test_auc': float(gpu_auc),
    'gpu_trained': True,
    'normalization': 'log1p on amounts, clip 0-1 on ratios',
    'benchmark': {
        'dataset_size': len(df),
        'cpu_time': round(cpu_time, 1),
        'gpu_time': round(gpu_time, 1),
        'speedup': round(speedup, 1),
        'cpu_auc': round(cpu_auc, 4),
        'gpu_auc': round(gpu_auc, 4),
        'gpu_type': 'NVIDIA Tesla T4',
        'xgboost_version': xgb.__version__
    }
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

feature_schema = {
    'n_features': len(features),
    'target': TARGET,
    'id_column': ID_COL,
    'categorical_features': CAT_COLS,
    'normalization': {
        'log_cols': LOG_COLS,
        'ratio_cols': RATIO_COLS
    },
    'features': [{'name': f, 'type': 'float', 'description': f} for f in features]
}
with open('feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)

print('Model exported!')
print(json.dumps(metadata, indent=2))

In [ ]:
# SHAP - Explicabilité
import shap
explainer = shap.TreeExplainer(model_gpu)
shap_values = explainer.shap_values(X_test.iloc[:100])
shap.summary_plot(shap_values, X_test.iloc[:100], show=False)
print('SHAP computed successfully')

In [ ]:
# Télécharger les artefacts
import os
for f in ['xgboost_model.json', 'feature_order.json', 'model_metadata.json', 'feature_schema.json']:
    if os.path.exists(f):
        print(f'{f}: {os.path.getsize(f)} bytes')